In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn import preprocessing

In [2]:
# Load the file using the path exactly as it was saved
file_path = '../data/embeddings_2026-1-19_17-3-39_2176_32768_blocks.28.mlp.l3_16384_0_1.pt' 
data = torch.load(file_path, map_location=torch.device('cpu'))

# Extract the lists from the dictionary
ids = data["ids"]               # List of strings
# DeepPL files contains accession id without version number, so we need to erase it from our ids
ids = [item.split('.')[0] for item in ids] 
embeddings_list = data["embeddings"] # List of torch.Tensors

# Stack the list of tensors into one large tensor (N_samples x 4096)
embeddings_tensor = torch.stack(embeddings_list)

# Convert to Numpy array
loaded_embeds = embeddings_tensor.detach().cpu().numpy()
loaded_ids = np.array(ids)

print("Embeddings shape:", loaded_ids.shape)
print("Embeddings shape:", loaded_embeds.shape) 

df = pd.DataFrame(data=loaded_embeds[:,:],    # values
              index=loaded_ids[:],    # 1st column as index
              columns=[[f"feature_{i}" for i in range(len(loaded_embeds[0]))]]) 
# Take a look
df.head()

Embeddings shape: (20854,)
Embeddings shape: (20854, 4096)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_4086,feature_4087,feature_4088,feature_4089,feature_4090,feature_4091,feature_4092,feature_4093,feature_4094,feature_4095
NC_012663,47.00,-93.0,-69.5,96.5,102.0,-44.0,-122.0,44.00,-75.0,-268.0,...,-41.75,36.000,-59.00,-62.75,-59.5,82.0,80.5,-116.5,-74.5000,0.75000
NC_012663,46.00,-93.0,-75.0,94.5,97.5,-42.0,-120.0,41.75,-81.5,-260.0,...,-41.00,32.250,-61.00,-62.50,-61.5,78.0,80.5,-112.5,-78.5000,-6.78125
NC_012663,46.25,-94.0,-75.5,92.5,102.0,-39.0,-116.5,40.25,-84.0,-249.0,...,-41.50,30.125,-62.50,-65.50,-66.0,75.5,85.0,-110.5,-70.0000,-14.00000
NC_012663,50.25,-98.0,-63.5,91.5,119.0,-37.0,-114.0,45.00,-74.0,-246.0,...,-46.00,35.250,-63.25,-73.00,-72.0,87.0,103.5,-116.0,-42.5000,-10.25000
NC_012663,60.50,-88.5,-76.0,102.5,73.0,-43.5,-126.5,36.50,-80.0,-270.0,...,-52.75,37.500,-59.25,-45.00,-38.5,91.0,76.0,-113.5,-10.3125,-38.75000


In [3]:
# Path to the Excel file containing lifecycle information
excel_file_path = "../data/journal.pcbi.1012525.s001.xlsx"
# Training
sheet_names = pd.ExcelFile(excel_file_path).sheet_names
# Training dataset
training = pd.read_excel(excel_file_path, sheet_name=sheet_names[0])
# Test dataset
test = pd.read_excel(excel_file_path, sheet_name=sheet_names[1])

# Create a dictionary mapping Accession ID -> Corrected Lifecycle
lifecycle_map = {}
all_metadata = pd.concat([training, test])

for _, row in all_metadata.iterrows():
    acc = row['Accession number']
    raw_life = row['Lifecycle']
    clean_life = "lysogenic" if str(raw_life).strip().lower() in ["lysogneic", "lysogenic"] else "lytic"
    lifecycle_map[acc] = clean_life

# Create aligned list
aligned_lifecycles = []
for accession in df.index:
    if accession in lifecycle_map:
        aligned_lifecycles.append(lifecycle_map[accession])
    else:
        # Warning only once per missing ID to avoid clutter
        print(f"Warning: Accession {accession} not found. Defaulting to lytic.")
        aligned_lifecycles.append("lytic") 

aligned_lifecycles = np.array(aligned_lifecycles)
print(f"Aligned labels length: {len(aligned_lifecycles)}")

Aligned labels length: 20854


In [4]:
# Set global random state for following models
RANDOM_STATE = 0

train_ids = set(training['Accession number'].astype(str).str.strip())
test_ids = set(test['Accession number'].astype(str).str.strip())

# Prepare feature and labels
train_indices = []
test_indices = []
y_train = []
y_test = []

for i, accession in enumerate(df.index):
    label = lifecycle_map.get(accession, "lytic")
    
    if accession in train_ids:
        train_indices.append(i)
        y_train.append(label)
    elif accession in test_ids:
        test_indices.append(i)
        y_test.append(label)


X_train = df.iloc[train_indices]
X_test = df.iloc[test_indices]
y_train = np.array(y_train)
y_test = np.array(y_test)

# Normalize data
normalizer = preprocessing.Normalizer()
X_train = normalizer.transform(X_train)
X_test = normalizer.transform(X_test)

print(f"Data Split Loaded: {len(X_train)} Train, {len(X_test)} Test")


Data Split Loaded: 17226 Train, 3628 Test


In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, 
                             matthews_corrcoef, make_scorer)
from sklearn.model_selection import TunedThresholdClassifierCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# We tell the tuner: "Find the threshold that maximizes F1 score for 'lysogenic'"
f1_scorer = make_scorer(f1_score, pos_label="lysogenic")

# Define function to print metrics for each model
def compute_and_print_metrics(model_name, y_true, y_pred, pos_label="lysogenic"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    neg_label = "lytic" if pos_label == "lysogenic" else "lysogenic"
    cm = confusion_matrix(y_true, y_pred, labels=[neg_label, pos_label])
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    print(f"\n{model_name} RESULTS")
    print("-" * 30)
    print(f"Sensitivity (Recall): {sensitivity:.4f}")
    print(f"Specificity:          {specificity:.4f}")
    print(f"Accuracy:             {acc:.4f}")
    print(f"F1-Score:             {f1:.4f}")
    print(f"MCC:                  {mcc:.4f}")
    print("-" * 30)
    print("Confusion Matrix:")
    print(f"                 Pred {neg_label}  Pred {pos_label}")
    print(f"Actual {neg_label:9} {tn:5d}           {fp:5d}")
    print(f"Actual {pos_label:9} {fn:5d}           {tp:5d}")


In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, TunedThresholdClassifierCV
from sklearn.metrics import make_scorer, f1_score

param_grid_hgb = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_iter': [200, 500, 1000],
    'max_depth': [3, 5, 10],
    'l2_regularization': [0.0, 0.1, 1]
}

# We use class_weight='balanced' natively (no manual calculation needed)
hgb_base = HistGradientBoostingClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    scoring='loss' # Optimizes log-loss internally
)

# Define Scorer (Optimizing F1 for Lysogenic)
# Since HistGradientBoosting accepts strings, we don't strictly need LabelEncoder for y,
# but using the scorer with pos_label ensures clarity.
scorer = make_scorer(f1_score, pos_label="lysogenic")

print("Starting HistGradientBoosting Grid Search...")
grid_search_hgb = GridSearchCV(
    hgb_base, 
    param_grid_hgb, 
    cv=5, 
    scoring=scorer, 
    n_jobs=-1, # Adjust based on your available cores
    verbose=1
)

grid_search_hgb.fit(X_train, y_train)

best_hgb = grid_search_hgb.best_estimator_
print(f"Best HGB Params: {grid_search_hgb.best_params_}")

# Tune decision threshold on the optimized model
print("Tuning Decision Threshold...")
hgb_final = TunedThresholdClassifierCV(best_hgb, scoring=scorer, cv=5)
hgb_final.fit(X_train, y_train)

print(f"HGB Optimal Threshold: {hgb_final.best_threshold_:.3f}")

# HistGradientBoosting in sklearn handles string labels natively!
y_pred_hgb = hgb_final.predict(X_test)

compute_and_print_metrics("HistGradientBoosting (Sklearn's XGBoost)", y_test, y_pred_hgb)

Starting HistGradientBoosting Grid Search...
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best HGB Params: {'l2_regularization': 1, 'learning_rate': 0.1, 'max_depth': 10, 'max_iter': 200}
Tuning Decision Threshold...
HGB Optimal Threshold: 0.505

HistGradientBoosting (Sklearn's XGBoost) RESULTS
------------------------------
Sensitivity (Recall): 0.7627
Specificity:          0.9775
Accuracy:             0.9283
F1-Score:             0.8296
MCC:                  0.7896
------------------------------
Confusion Matrix:
                 Pred lytic  Pred lysogenic
Actual lytic      2735              63
Actual lysogenic   197             633


In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, f1_score
from joblib import dump

scorer = make_scorer(f1_score, pos_label="lysogenic")


# Best HGB Params: {'l2_regularization': 0.1, 'learning_rate': 0.2, 'max_depth': 5, 'max_iter': 500}
hgb_norm = HistGradientBoostingClassifier(
    class_weight='balanced',
    random_state=0,
    scoring='loss',
    l2_regularization=1,
    learning_rate=0.1,
    max_depth=10,
    max_iter=200
)

hgb_norm.fit(X_train, y_train)

y_pred_test_hgb_norm = hgb_norm.predict(X_test)

print("Tuning Decision Threshold...")
hgb_norm = TunedThresholdClassifierCV(hgb_norm, scoring=scorer, cv=5)
hgb_norm.fit(X_train, y_train)

print(f"HGB Optimal Threshold: {hgb_norm.best_threshold_:.3f}")

# # Save the model
# dump(hgb_norm, "hgb_model.joblib")
# # HistGradientBoosting in sklearn handles string labels natively!
# y_pred_hgb = hgb_norm.predict(X_test)


compute_and_print_metrics("HistGradientBoosting (Sklearn's XGBoost)", y_test, y_pred_test_hgb_norm)

Tuning Decision Threshold...
HGB Optimal Threshold: 0.505

HistGradientBoosting (Sklearn's XGBoost) RESULTS
------------------------------
Sensitivity (Recall): 0.7627
Specificity:          0.9775
Accuracy:             0.9283
F1-Score:             0.8296
MCC:                  0.7896
------------------------------
Confusion Matrix:
                 Pred lytic  Pred lysogenic
Actual lytic      2735              63
Actual lysogenic   197             633


It seems that repetition with embeddings for both strands does not improve performance.